# PostgresDB Manager Usage Examples

This notebook demonstrates a practical `PostgreSQL` workflow with `py-dockerdb`, focusing on fast local setup, predictable lifecycle control, and reproducible execution from a single Python interface. Instead of manually wiring container commands for each run, you configure once, start the service, validate connectivity, and then execute database operations in a consistent sequence.

If you are comparing engines, this pattern helps you isolate query behavior and schema differences without rewriting orchestration code. Official reference: https://www.postgresql.org/docs/.

## Table Of Contents

- [Table Of Contents](#table-of-contents)
- [Prerequisites](#prerequisites)
- [1. Setup](#1-setup)
  - [Import Dependencies](#import-dependencies)
- [2. Creating a PostgreSQL Database Instance](#2-creating-a-postgresql-database-instance)
- [3. Start the Database](#3-start-the-database)
- [4. Connect and Run SQL Queries](#4-connect-and-run-sql-queries)
  - [Creating a Table](#creating-a-table)
  - [Insert Data](#insert-data)
  - [Query Data](#query-data)
  - [Run More Complex Queries](#run-more-complex-queries)
- [5. Using Regular Python to Access the Database](#5-using-regular-python-to-access-the-database)
- [6. Clean Up](#6-clean-up)
- [7. Conclusion](#7-conclusion)
- [8. Conclusion](#8-conclusion)

## Prerequisites

No environment variables are required unless you intentionally override the defaults used in this notebook.

Additional prerequisites:
- Docker must be installed and running before you call `create_db()`.
- Install project dependencies (for example, `pip install -e ".[test]"` or `pip install py-dockerdb`).
- Run cells top-to-bottom so container lifecycle state remains consistent.

Provider documentation: https://www.postgresql.org/docs/
Typical setup guide: https://www.postgresql.org/download/
Docker image setup reference: https://hub.docker.com/_/postgres

Example datasets you can use to populate this database:
- DVD Rental sample database: https://www.postgresqltutorial.com/postgresql-getting-started/postgresql-sample-database/
- Pagila sample dataset: https://github.com/devrimgunduz/pagila

## 1. Setup

### Install Required Packages

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
!pip install ipython-sql==0.5.0 prettytable==3.8.0
!pip uninstall py-dockerdb -y

### Import Dependencies

This subsection details `import dependencies` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path().cwd().parent))
print(sys.path[0])

This step executes code block `5` and is required for the next stage of the workflow.

Run this only after the previous step completed successfully, because downstream cells assume the resulting object, container state, or connection is already available.

In [ ]:
import uuid
from pathlib import Path
from docker_db.dbs.postgres_db import PostgresConfig, PostgresDB

# For SQL cell magic
%load_ext sql

## 2. Creating a PostgreSQL Database Instance

Now, let's set up the PostgresDB configuration:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
import tempfile
import os

temp_dir = Path("tmp")
temp_dir.mkdir(exist_ok=True)

container_name = f"demo-postgres-{uuid.uuid4().hex[:8]}"
init_script_path = Path("configs", "postgres", "initdb.sh")
init_script_path.exists()

This step executes code block `8` and is required for the next stage of the workflow.

Run this only after the previous step completed successfully, because downstream cells assume the resulting object, container state, or connection is already available.

In [ ]:
from utils import display_bash_script
display_bash_script(init_script_path)

This step executes code block `9` and is required for the next stage of the workflow.

Run this only after the previous step completed successfully, because downstream cells assume the resulting object, container state, or connection is already available.

In [ ]:
# Create a configuration for our database
config = PostgresConfig(
    user="demouser",
    password="demopass",
    database="demodb",
    project_name="demo",
    container_name=container_name,
    workdir=temp_dir.absolute(),
    retries=20,
    delay=3,
    init_script=init_script_path,
    env_vars={"YourEnvVar": "TestVar"},
)

# Initialize the database manager
db_manager = PostgresDB(config)

## 3. Start the Database

We'll now create and start the database:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
# Create and start the database
db_manager.create_db()
print(f"Database started successfully in container '{container_name}'")
print(f"Connection details:")
print(f"  Host: {config.host}")
print(f"  Port: {config.port}")
print(f"  User: {config.user}")
print(f"  Database: {config.database}")

## 4. Connect and Run SQL Queries

Now that our database is running, let's connect to it using SQL cell magic:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
# Define the connection string for SQL magic
conn_string = db_manager.connection_string(sql_magic=True)
%sql $conn_string

This step executes code block `14` and is required for the next stage of the workflow.

Run this only after the previous step completed successfully, because downstream cells assume the resulting object, container state, or connection is already available.

In [ ]:
%%sql
SELECT * FROM test_table;

### Creating a Table

This subsection details `creating a table` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
%%sql
CREATE TABLE demo_users (
    id SERIAL PRIMARY KEY,
    username VARCHAR(50) UNIQUE NOT NULL,
    email VARCHAR(100) UNIQUE,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

### Insert Data

This subsection details `insert data` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
%%sql
INSERT INTO demo_users (username, email) VALUES
    ('alice', 'alice@example.com'),
    ('bob', 'bob@example.com'),
    ('charlie', 'charlie@example.com');

### Query Data

This subsection details `query data` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
%%sql
SELECT * FROM demo_users;

### Run More Complex Queries

This subsection details `run more complex queries` before the next code cell.

This subsection introduces the next concrete operation and clarifies why it matters before you run the following code cell.

In [ ]:
%%sql
-- Create another table for demonstration
CREATE TABLE demo_posts (
    id SERIAL PRIMARY KEY,
    user_id INTEGER REFERENCES demo_users(id),
    title VARCHAR(100) NOT NULL,
    content TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

-- Insert some posts
INSERT INTO demo_posts (user_id, title, content) VALUES
    (1, 'Alice First Post', 'Hello world from Alice!'),
    (1, 'Alice Second Post', 'Another post from Alice'),
    (2, 'Bob Introduction', 'Hi, this is Bob!'),
    (3, 'Charlie Notes', 'Some notes from Charlie');
    
-- Query with JOIN
SELECT u.username, p.title, p.content
FROM demo_users u
JOIN demo_posts p ON u.id = p.user_id
ORDER BY u.username, p.created_at;

## 5. Using Regular Python to Access the Database

You can also interact with the database using Python code and psycopg2:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
import psycopg2
from psycopg2.extras import RealDictCursor

# Connect to the database
conn = db_manager.connection

# Create a cursor
cur = conn.cursor()

# Execute a query
cur.execute("SELECT COUNT(*) as post_count FROM demo_posts")
result = cur.fetchone()
print(f"Total number of posts: {result['post_count']}")

# Group by query
cur.execute("""
    SELECT u.username, COUNT(p.id) as post_count 
    FROM demo_users u
    LEFT JOIN demo_posts p ON u.id = p.user_id
    GROUP BY u.username
    ORDER BY post_count DESC
""")

print("\nPost count by user:")
for row in cur.fetchall():
    print(f"  {row['username']}: {row['post_count']} posts")

# Close the connection
cur.close()
conn.close()

## 6. Clean Up

When you're done with the database, you can delete it:

This section provides context for the next executable step so the workflow remains reproducible and easy to debug.

In [ ]:
# Delete the database container
db_manager.delete_db(running_ok=True)
print(f"Database container '{container_name}' deleted")

## 7. Conclusion

This notebook demonstrated how to:

1. Configure and create a PostgreSQL database with `PostgresDB`
2. Connect to the database using SQL cell magic
3. Execute SQL queries on the database
4. Clean up the database when finished

The `PostgresDB` manager provides a convenient way to spin up PostgreSQL instances in Docker containers for development, testing, or demonstration purposes.

For production use, align this tutorial flow with your team standards: credential handling, migration strategy, backup policy, and monitoring. When needed, start from the provider setup docs (https://www.postgresql.org/download/) and validate against real datasets before benchmarking query performance.